This is a **CLEAN** but vivid version of codes (along with visualizations) 
for analyzing the experiment ***freight collaboration-chessboard*** results

# Required packages

In [ ]:
from dataclasses import dataclass
from enum import Enum
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from itertools import product
import matplotlib.pyplot as plt
import seaborn as sns
# Use repo-relative path so it works on other machines
notebook_dir = Path.cwd() / "python" / "test"
if notebook_dir.exists() and str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))
import matsim
from pointpats import PointPattern, PoissonPointProcess
from pointpats.distance_statistics import g, f, k, l, j
import matsim_output_reader
import metric_anls
import figure_plot 
import agg_anls
import spatial_anls
# --- reload the module to reflect any changes made during development ---
import importlib
importlib.reload(matsim_output_reader)
importlib.reload(metric_anls)
importlib.reload(figure_plot)
importlib.reload(agg_anls)
importlib.reload(spatial_anls)

# Configuration

In [ ]:
# Resolve analysis path relative to repo root
repo_root = notebook_dir.parents[1]  # .../matsim-libs-2024
anls_path = repo_root / "output" / "chessboardCarrierReceiverCollab"

class DepotLocation(Enum):
    INSIDE = 'center'
    OUTSIDE = 'left'

class ReceiverDistribution(Enum):
    DISPERSED = 'DISPERSED'
    CLUSTERED = 'CLUSTERED'
    RANDOM = 'FULLY_RANDOM'

ALLOCATION_FACTOR = 0.8
ALLOCATION_FACTOR_LIST = [x / 10 for x in range(1, 10)]  # 0.1 to 0.9 with step of 0.1
#--- Penalty ---#
PENALTY_LIST = [0, 0.0003, 0.0008, 0.0014, 0.0028, 0.0056,
                0.0098, 0.014, 0.0167, 0.0222, 0.028, 0.0333, 0.0417]
PENALTY_LIST_SCALE = [round(x * 3600) for x in PENALTY_LIST]  # scale to avoid float precision issues
PENALTY_LIST_SCALE[PENALTY_LIST.index(0.028)] = 100 
PENALTY_DICT = {k: v for k, v in zip(PENALTY_LIST, PENALTY_LIST_SCALE)}
#--- Penalty ---#

TOTAL_INSTANCES = 60


In [ ]:
# Configuration for batch processing
# INPUT_PATH = str(anls_path)
OUTPUT_PATH = str(repo_root / "data" / "freightChessboardRC" / "clean")
DISPERSED_OUTPUT_PATH = str(repo_root / "data" / "freightChessboardRC" / "cleanMoreDispersed")
MORE_AF_OUTPUT_PATH = str(repo_root / "data" / "freightChessboardRC" / "cleanMoreAF")
MORE_PEN_OUTPUT_PATH = str(repo_root / "data" / "freightChessboardRC" / "cleanMorePen")

# Define which scenarios to process
DEPOT_LOCATIONS = [DepotLocation.INSIDE.value, DepotLocation.OUTSIDE.value]
RECEIVER_DISTRIBUTIONS = [ReceiverDistribution.DISPERSED.value, ReceiverDistribution.CLUSTERED.value]
ORIGINAL_TW = (6, 7)  # Original time window in hours
LAST_ITER = 30  # Last iteration number

In [ ]:
#--- Generate keywords for each scenario ---#
# Example tuple: ("center", "dispersed", 0)
scenario_keywords = [
    (depot, receiver, penalty)
    for depot, receiver, penalty in product([DepotLocation.INSIDE.value, DepotLocation.OUTSIDE.value],
                                            [ReceiverDistribution.DISPERSED.value, ReceiverDistribution.CLUSTERED.value], 
                                            PENALTY_LIST)
]
scenario_keywords[:3]

In [ ]:
OUTPUT_FIG_PATH = repo_root / "data" / "freightChessboardRC" / "figures"
OUTPUT_FIG_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
network_dir = repo_root / "data" / "freightChessboardRC" / "output_network.xml.gz"
network = matsim.read_network(network_dir)
network_links = network.links
network_nodes = network.nodes

In [ ]:
# Build network graph once for efficiency
network_graph = agg_anls.build_network_graph(network_links, network_nodes)
print(f"Network graph: {network_graph.number_of_nodes()} nodes, {network_graph.number_of_edges()} edges")

In [ ]:
full_network_gdf = spatial_anls.network_graph_to_gdf(network_graph)
full_network_gdf

In [ ]:
central_area_network_gdf = spatial_anls.network_graph_to_gdf(
    network_graph,
    boundary=[2000, 2000, 7000, 7000])
central_area_network_gdf

# Read all_instance_metric df

In [ ]:
all_metrics_df = pd.read_csv(os.path.join(OUTPUT_PATH, 'all_scenarios_metrics.csv.gz'), compression='gzip')
all_metrics_df

In [ ]:
'''Add more AF instances (af->(0.1-0.4)) '''
ins_more_af_df = pd.read_csv(os.path.join(MORE_AF_OUTPUT_PATH, 'all_scenarios_metrics.csv.gz'), compression='gzip') 
ins_more_af_df

In [ ]:
''' add more penalty instances (0.1-0.4) with the AF 0.8 '''
ins_more_pen_af80 = pd.read_csv(os.path.join(MORE_PEN_OUTPUT_PATH, 'all_scenarios_metrics.csv.gz'), compression='gzip')
ins_more_pen_af80

In [ ]:
all_metrics_df = pd.concat([all_metrics_df, ins_more_af_df, ins_more_pen_af80], ignore_index=True)
all_metrics_df

In [ ]:
all_metrics_df = all_metrics_df[~all_metrics_df['instance'].between(20,29)]
all_metrics_df

In [ ]:
# aggregate_shipment_metrics_from_clean_folder function
clean_data_path = repo_root / 'data/freightChessboardRC/clean'
shipment_metrics_df = agg_anls.aggregate_shipment_metrics_from_clean_folder(str(clean_data_path), verbose=True)
shipment_metrics_df

In [ ]:
shipment_metrics_df = shipment_metrics_df[~shipment_metrics_df['instance_id'].between(20, 29)]
shipment_metrics_df['diff_fleet_size'] = shipment_metrics_df['final_fleet_size'] - shipment_metrics_df['iter0_fleet_size']

In [ ]:
all_metrics_df = pd.merge(all_metrics_df, 
                          shipment_metrics_df[['instance_id', 'allocation_factor', 'depot_location', 'receiver_distribution', 'penalty','diff_fleet_size', 'final_fleet_size']],
                          left_on=['instance', 'allocation_factor', 'depot_location', 'receiver_distribution', 'penalty'], 
                          right_on=['instance_id', 'allocation_factor', 'depot_location', 'receiver_distribution', 'penalty'], 
                          how='left')
all_metrics_df

In [ ]:
all_metrics_df['penalty_std'] = all_metrics_df['penalty'].map(PENALTY_DICT)
all_metrics_df 

## Rectify the dispersed related records

In [ ]:
''' drop all records with receiver_distribution == 'DISPERSED' '''
all_metrics_df = all_metrics_df[all_metrics_df['receiver_distribution'] != 'DISPERSED'].reset_index(drop=True)
all_metrics_df.shape

In [ ]:
ins_dispersed_metrics_df = pd.read_csv(os.path.join(DISPERSED_OUTPUT_PATH, 'all_scenarios_metrics.csv.gz'), compression='gzip')
ins_dispersed_metrics_df

In [ ]:
CORRECT_DISPERSED_INS_LIST = ins_dispersed_metrics_df['instance'].unique().tolist()
CORRECT_DISPERSED_INS_LIST

In [ ]:
''' Concat the dispersed metrics to the main metrics DataFrame '''
all_metrics_df = pd.concat([all_metrics_df, ins_dispersed_metrics_df], ignore_index=True)
all_metrics_df.shape

In [ ]:
all_metrics_df = all_metrics_df[all_metrics_df['instance'].isin(CORRECT_DISPERSED_INS_LIST)].reset_index(drop=True)
all_metrics_df.shape

## Split to specific dfs

In [ ]:
all_metrics_df['collaboration_rate'] = all_metrics_df['num_collaborative_receivers'] / 10
all_metrics_df

In [ ]:
''' Random dfs '''
all_metrics_random_df = all_metrics_df[all_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value]
all_metrics_center_random_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value)]
all_metrics_outside_random_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value)]


In [ ]:
all_metrics_center_clustered_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)]
all_metrics_center_dispersed_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)]
all_metrics_outside_clustered_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)]
all_metrics_outside_dispersed_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)]

In [ ]:
all_metrics_non_random_df = all_metrics_df[all_metrics_df['receiver_distribution'] != ReceiverDistribution.RANDOM.value]

# Random scenario analysis
1. To demonstrate the effectiveness of the freight collaboration
2. To point out the possible effects/impacts of the spatial distribution of receivers

## Penalty sweep
The `allocation_factor` is fixed as 0.6

### Collaboration rate (box plot)

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_random_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(8, 5),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af60_all_random_receivers.png',
)

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_center_random_df.query("allocation_factor == 0.8"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=True,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 4),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af80_center_random_receivers.png',
)

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_outside_random_df.query("allocation_factor == 0.8"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    box_alpha=0.9,
    se_box_color_for_lines=True,
    # Display options
    flier_marker='X',
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    # Show scatter points with jitter
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    #--- Violin plot ---
    show_violin=True,
    violin_alpha=0.3, 
    violin_width=0.9,
    #--- Grid ---
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 4),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af80_outside_random_receivers.png',
)

In [ ]:
metric_anls.compute_ashmans_d(
    df=all_metrics_center_random_df.query("penalty_std == 20"),
    col_name='collaboration_rate',
)

### Collaboration rate (histogram)
Only plotting those groups with large variance

#### All-af60-p20

In [ ]:
figure_plot.hist_plot(all_metrics_random_df[(all_metrics_random_df['allocation_factor'] == 0.6) & (all_metrics_random_df['penalty_std'] == 20)],
                      col1='collaboration_rate',
                      n_bins=8,
                      figure_size=(4,2.5),
                      ylabel='Density',
                      xlabel='Collaboration rate',
                      label_size=14,
                      hide_labels=False,
                      hide_legends=True,
                      alphas=(0.9, 0.9),
                      colors=("#8dadc3", '#f6bdb1'),
                      #---fitting---
                      fitting_method='kde',
                      kde_bw=0.3,
                      figure_folder=OUTPUT_FIG_PATH,  
                      filename='collab_rate_distribution_all_random_af60_p20euro.png'
                      )
metric_anls.compute_ashmans_d(
    df=all_metrics_random_df[(all_metrics_random_df['allocation_factor'] == 0.6) & (all_metrics_random_df['penalty_std'] == 20)],
    col_name='collaboration_rate',
)

#### All-af60-p35

In [ ]:
figure_plot.hist_plot(all_metrics_random_df[(all_metrics_random_df['allocation_factor'] == 0.6) & (all_metrics_random_df['penalty_std'] == 35)],
                      col1='collaboration_rate',
                      n_bins=8,
                      figure_size=(4,2.5),
                      ylabel='Density',
                      xlabel='Collaboration rate',
                      label_size=14,
                      hide_labels=False,
                      hide_legends=True,
                      alphas=(0.9, 0.9),
                      colors=("#8dadc3", '#f6bdb1'),
                      #---fitting---
                      fitting_method='gmm',
                      figure_folder=OUTPUT_FIG_PATH,  
                      filename='collab_rate_distribution_all_random_af60_p35euro.png'
                      )

#### All-af60-p50

In [ ]:
figure_plot.hist_plot(all_metrics_random_df[
    (all_metrics_random_df['allocation_factor'] == 0.6) & 
    (all_metrics_random_df['penalty_std'] == 50)
    ],
                      col1='collaboration_rate',
                      n_bins=5,
                      figure_size=(3.5,2.7),
                      ylabel='Density',
                      xlabel='Collaboration rate',
                      label_size=10,
                      hide_labels=False,
                      hide_legends=True,
                      alphas=(0.9, 0.9),
                      colors=("#8dadc3", '#f6bdb1'),
                      #---fitting---
                      fitting_method='kde',
                      figure_folder=OUTPUT_FIG_PATH,  
                      filename='collab_rate_distribution_all_random_af60_p50euro.png'
                      )

### Total cost savings

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_random_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='total_cost_savings',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Total cost savings',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4, 3),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='total_cost_savings_by_penalty_af60_all_random_receivers.png',
)

In [ ]:
all_metrics_random_df.query("allocation_factor == 0.8").plot.scatter(x='penalty_std', y='total_cost_savings')

### fleet size

In [ ]:
_df = all_metrics_random_df.query("allocation_factor == 0.6")
_df['diff_fleet_size'] = _df['diff_fleet_size'].map(lambda x: 0 if x >=1 else x)  # Map all values >= 1 to 0, keep others unchanged
_df['diff_fleet_size'] = _df['diff_fleet_size'] * -1

_colors = figure_plot.generate_smooth_colors(
    input_colors=["#abd1e9", "#003b64"],
    n_req_colors=4
)
figure_plot.stacked_proportion_plot(_df,
                                    figure_size=(12,4),
                                    dpi=350,
                                     bin_col='penalty_std',
                                     value_col='final_fleet_size',
                                    # n_bins=5,
                                     bin_method='unique',
                                    #  custom_bins=[0.4, 0.6, 0.8, 1.0],
                                    #  colors=['#757575',"#abd1e9", "#458CBC", 
                                    #         #  "#005681DF",
                                    #          "#003b64"],
                                     colors=_colors,
                                     xlabel='Cost of collaboration (euros/hour)',
                                     ylabel='Proportion (%)',
                                     label_size=14,
                                     show_counts=False,
                                     count_fontsize=12,
                                     percentage_fontsize=12,
                                     # Legend settings
                                     legend_bbox=(0.5, 1.4),
                                     legend_ncol=3,
                                     legend_title='Fleet size',
                                     show_legend=False,
                                     # Output
                                     figure_folder=OUTPUT_FIG_PATH,
                                     filename='all_random_af60_stacked_proportion_fleet_size_vs_penalty.png'
                                     )

### Scores

In [ ]:
_df = all_metrics_random_df.query("allocation_factor == 0.8").groupby('penalty_std')['final_carrier_score'].agg(['mean', 'std', 'count'])
_df.plot.line( y='mean')

### VKT

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_random_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='VKT_km',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='VKT',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4, 3),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='VKT_by_penalty_af60_all_random_receivers.png',
)

### TKT

In [ ]:
_tkt_df = all_metrics_random_df.query("allocation_factor == 0.6")
_tkt_df['TKT_tonkm'] = _tkt_df['TKT_tonkm'] / 1000  # Scale to thousand ton-km

figure_plot.box_plot(
    data_list=_tkt_df,
    cat_col='penalty_std',
    col_name='TKT_tonkm',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Ton-km travelled',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4, 3),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='TKT_by_penalty_af60_all_random_receivers.png',
)

## Allocation sweep

### Collaboration rate

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_random_df.query("penalty_std == 20"),
    cat_col='allocation_factor',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Allocation factor',
    #--- Violin plot ---
    show_violin=True,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 4),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    # filename='collaboration_rate_by_penalty_af80_all_random_receivers.png',
)

## Joint-af-pen

In [ ]:
all_metrics_random_df[all_metrics_random_df['penalty_std']>100]

In [ ]:
joint_af_pen_df = all_metrics_random_df.copy(deep=True)
joint_af_pen_df = joint_af_pen_df.query("penalty_std >0 and penalty_std <=100")


### Heatmap for collab

In [ ]:
_colors = figure_plot.generate_smooth_colors(
    input_colors=["#F7F2F2","#702F35"],
    n_req_colors=10
)
figure_plot.heatmap_plot(
    joint_af_pen_df,
    x_col='penalty_std',
    y_col='allocation_factor',
    value_col='collaboration_rate',
    agg_method='mean',
    cmap=_colors,
    #--- Annotation options ---
    annot=False,
    annot_size=12,
    # annot_color='black',
    annot_fmt='.2f',
    xlabel='Cost of collaboration (euros/hour)',
    ylabel='Allocation factor',
    label_size=14,
    #--- Output ---
    cbar=False,
    figure_size=(6, 6),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='heatmap_collaboration_rate_by_penalty_and_allocation_factor_random_receivers.png',
    # tick_label_map=PENALTY_DICT,
)

### Change rate lines

In [ ]:
collab_value_df = all_metrics_random_df.query("penalty_std <=100").groupby(['allocation_factor', 'penalty_std'])['collaboration_rate'].mean().unstack()
collab_value_df.fillna(1, inplace=True)  # Fill pen-0 NaN values with 1
collab_value_df

In [ ]:
change_value_df = collab_value_df.shift(1, axis=1) - collab_value_df
change_value_df = change_value_df.iloc[:, 1:]  # Remove the first column which will be NaN after shift
# # change it to percentage change
# change_value_df = change_value_df.apply(lambda x: x / collab_value_df[x.name] * 100)
_list = [0.2, 0.4, 0.6, 0.8]
# _list = [0.1, 0.3, 0.5, 0.7, 0.9]
change_value_df = change_value_df[change_value_df.index.isin(_list)]
change_value_df

In [ ]:
''' Plot the lines for each allocation factor-penalty change'''
_colors= figure_plot.generate_smooth_colors(
    input_colors=["#C7B7F8","#292252"],  # #463A8B
    n_req_colors=change_value_df.shape[0]
)
figure_plot.plot_change_rate(
    pivot_df=change_value_df,
    xlabel='Cost of collaboration (euros/hour)',
    ylabel='Collaboration rate drop',
    label_size=14,
    legend_title='Allocation factor',
    legend_frameon=False,
    colors=_colors,
    # alpha=0.6,
     #--- Output ---
    show_legend=True,
    figure_size=(5, 3),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='change_rate_collaboration_rate_by_penalty_random_receivers.png',
)


In [ ]:
sns.color_palette("tab20c")

In [ ]:
collab_value_df = all_metrics_random_df.query("penalty_std <= 100").groupby(['penalty_std', 'allocation_factor',])['collaboration_rate'].mean().unstack()
collab_value_df = collab_value_df.apply(lambda x: round(x,2))
collab_value_df[0.0] = 0
collab_value_df = collab_value_df.sort_index(axis=1) 
change_value_df = collab_value_df - collab_value_df.shift(1, axis=1)
change_value_df = change_value_df.iloc[:, 1:] 
change_value_df = change_value_df.applymap(lambda x: 0 if x <= 0 else x)  # Map all positive changes to 0, keep negative changes unchanged
change_value_df = change_value_df[change_value_df.index.to_series().between(10, 50)]

change_value_df

In [ ]:
_colors = figure_plot.generate_smooth_colors(
    input_colors=["#b6dbe3","#1D483F"],
    n_req_colors=change_value_df.shape[1]
)
_colors = ["#badad3", '#97b0aa', "#5f7570", '#1D483F']
figure_plot.plot_change_rate(
    # data_df=all_metrics_random_df[all_metrics_random_df['penalty_std'].between(10, 60)],
    # group_col='penalty_std',
    # x_col='allocation_factor',
    # value_col='collaboration_rate',
    # agg_method='mean',
    pivot_df=change_value_df,
    xlabel='Allocation factor',
    ylabel='Collaboration rate rise',
    label_size=14,
    legend_title='Cost of collab.',
    legend_frameon=False,
    colors=_colors,
    #--- Output ---
    # show_legend=False,
    show_legend=True,
    figure_size=(5, 3),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='change_rate_collaboration_rate_by_allocation_factor_random_receivers.png',
)

### Keep collaboration rate lines

In [ ]:
''' Full collaborate '''
_colors = figure_plot.generate_smooth_colors(
    input_colors=["#F7F2F2","#702F35"],
    n_req_colors=10
)
figure_plot.heatmap_plot(
    joint_af_pen_df,
    x_col='penalty_std',
    y_col='allocation_factor',
    value_col='collaboration_rate',
    agg_method='mean',
    annot=False,
    cmap=_colors,
    # --- Highlight cells with value >= 0.8 ---
    cap_value=0.91,
    cap_edgecolor="#C72323",       # border colour
    cap_linewidth=2,         # border width
    cap_linestyle='--',         # '-', '--', ':', etc.
    cap_compare='>=',          # '>=', '>', '<=', '<', '==', '!='
    # --- cap line ---
    cap_outer_only=True,       # draw only the outer boundary of contiguous highlighted regions
    cap_fill=True,           # optional translucent fill
    cap_fill_color="#F08787",
    cap_fill_alpha=0.4,
    #---output---
    cbar=False,
    xlabel='Cost of collaboration (euros/hour)',
    ylabel='Allocation factor',
    label_size=14,
    figure_size=(4, 3.5),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='heatmap_collaboration_rate_by_penalty_and_allocation_factor_random_receivers_with_highlight_greater1.png',

)

In [ ]:
_colors = figure_plot.generate_smooth_colors(
    input_colors=["#F7F2F2","#702F35"],
    n_req_colors=10
)
figure_plot.heatmap_plot(
    joint_af_pen_df,
    x_col='penalty_std',
    y_col='allocation_factor',
    value_col='collaboration_rate',
    agg_method='mean',
    annot=False,
    cmap=_colors,
    # --- Highlight cells with value >= 0.8 ---
    cap_value=0.69,
    cap_edgecolor="#C72323",       # border colour
    cap_linewidth=2,         # border width
    cap_linestyle='--',         # '-', '--', ':', etc.
    cap_compare='>=',          # '>=', '>', '<=', '<', '==', '!='
    # --- cap line ---
    cap_outer_only=True,       # draw only the outer boundary of contiguous highlighted regions
    cap_fill=True,           # optional translucent fill
    cap_fill_color="#F08787",
    cap_fill_alpha=0.4,
    #---output---
    cbar=False,
    xlabel='Cost of collaboration (euros/hour)',
    ylabel='Allocation factor',
    label_size=14,
    figure_size=(4, 3.5),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='heatmap_collaboration_rate_by_penalty_and_allocation_factor_random_receivers_with_highlight_greater70.png',

)

In [ ]:
_colors = figure_plot.generate_smooth_colors(
    input_colors=["#F7F2F2","#702F35"],
    n_req_colors=10
)
figure_plot.heatmap_plot(
    joint_af_pen_df,
    x_col='penalty_std',
    y_col='allocation_factor',
    value_col='collaboration_rate',
    agg_method='mean',
    annot=False,
    cmap=_colors,
    # --- Highlight cells with value >= 0.5 ---
    cap_value=0.50,
    cap_edgecolor="#C72323",       # border colour
    cap_linewidth=2,         # border width
    cap_linestyle='--',         # '-', '--', ':', etc.
    cap_compare='>=',          # '>=', '>', '<=', '<', '==', '!='
    # --- cap line ---
    cap_outer_only=True,       # draw only the outer boundary of contiguous highlighted regions
    cap_fill=True,           # optional translucent fill
    cap_fill_color="#F08787",
    cap_fill_alpha=0.4,
    #---output---
    cbar=False,
    xlabel='Cost of collaboration (euros/hour)',
    ylabel='Allocation factor',
    label_size=14,
    figure_size=(4, 3.5),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='heatmap_collaboration_rate_by_penalty_and_allocation_factor_random_receivers_with_highlight_greater50.png',

)

## Analyse random scenario before-and-after operation

In [ ]:
all_metrics_random_20euro_af60_df = all_metrics_random_df[(all_metrics_random_df['penalty'] == 0.0056) & (all_metrics_random_df['allocation_factor'] == 0.6)]
all_metrics_random_20euro_af60_df

In [ ]:
print("the mean collaboration rate for 20 euro penalty is:", all_metrics_random_20euro_af60_df['collaboration_rate'].mean())
print("the max collaboration rate for 20 euro penalty is:", all_metrics_random_20euro_af60_df['collaboration_rate'].max())
print("the min collaboration rate for 20 euro penalty is:", all_metrics_random_20euro_af60_df['collaboration_rate'].min())

### VKT

In [ ]:
_vkt_df = all_metrics_outside_random_20euro_af80_df[all_metrics_outside_random_20euro_af80_df['instance'].between(0, 100)]
figure_plot.hist_plot(_vkt_df,
                      col1='VKT_km',
                      col2='iter0_VKT_km',
                      n_bins=10,
                      figure_size=(3.5,2.7),
                      ylabel = 'Density',
                      xlabel = 'VKT (km)',
                      label_size=14,
                      hide_labels=False,
                      hide_legends=True,
                      alphas=(0.9, 0.9),
                      colors=("#8dadc3", '#f6bdb1'),
                      figure_folder=OUTPUT_FIG_PATH,
                      filename='vkt_distribution_outside_random_20euro_penalty'
                      )
print("the mean reduction in VKT for 20 euro penalty is:",
       (_vkt_df['iter0_VKT_km'].mean() - _vkt_df['VKT_km'].mean()))

In [ ]:
_vkt_df = all_metrics_random_20euro_af60_df.copy(deep=True)
_vkt_df['vkt_reduction'] = _vkt_df['iter0_VKT_km'] - _vkt_df['VKT_km']

In [ ]:
figure_plot.scatter_regression_plot(_vkt_df,
                                    x_col='collaboration_rate',
                                    y_col='vkt_reduction',
                                    figure_size=(3.3,2.8),
                                    xlabel='Collaboration rate',
                                    ylabel='VKT reduction',
                                    label_size=14,
                                    annotation_fontsize=12,
                                    # title='Scatter plot with regression line: VKT vs Collaboration Rate',
                                    show_corr=True,
                                    add_regression=True,
                                    show_equation=False,
                                    show_r2=True,
                                    scatter_color='#458CBC',
                                    scatter_alpha=0.9,
                                    reg_color='#0b2c60',
                                    # Output
                                    figure_folder=OUTPUT_FIG_PATH,
                                    filename='scatter_regression_vkt_reduction_vs_collab_rate_all_random_20euro_af60.png'
                                    )

### VTT

In [ ]:
vtt_minutes_df = all_metrics_outside_random_20euro_af80_df.copy()
vtt_minutes_df[['VTT_seconds', 'iter0_VTT_seconds']] = (
    vtt_minutes_df[['VTT_seconds', 'iter0_VTT_seconds']] / 60
)
_vtt_minutes_df = vtt_minutes_df[vtt_minutes_df['instance'].between(0,100)]
figure_plot.hist_plot(_vtt_minutes_df,
                      col1='VTT_seconds',
                      col2='iter0_VTT_seconds',
                      n_bins=10,
                      figure_size=(3.5,2.7),
                      ylabel='Density',
                      xlabel='Travel time (min)',
                      label_size=14,
                      hide_labels=False,
                      hide_legends=True,
                      alphas=(0.9, 0.9),
                        colors=("#8dadc3", '#f6bdb1'),
                        figure_folder=OUTPUT_FIG_PATH,
                        filename='vtt_distribution_outside_random_20euro_penalty'
                      )
print("the mean reduction of VTT is: {}".format(
    -(_vtt_minutes_df['VTT_seconds'].mean() 
        - _vtt_minutes_df['iter0_VTT_seconds'].mean())) 
        )

### Ton-km travelled

In [ ]:
tkt_ton_df = all_metrics_outside_random_20euro_af80_df.copy()
tkt_ton_df[['TKT_tonkm', 'iter0_TKT_tonkm']] = (
    tkt_ton_df[['TKT_tonkm', 'iter0_TKT_tonkm']] / 1000
)
_tkt_ton_df = tkt_ton_df[tkt_ton_df['instance'].between(0,100)]
figure_plot.hist_plot(_tkt_ton_df,
                        col1='TKT_tonkm',
                        col2='iter0_TKT_tonkm',
                        n_bins=9,
                        figure_size=(3.3,2.8),
                        ylabel='Density',
                        xlabel='Ton-km travelled',
                        hide_labels=False,
                        hide_legends=True,
                        alphas=(0.9, 0.9),
                        colors=("#8dadc3", '#f6bdb1'),
                        label_size=14,
                        figure_folder=OUTPUT_FIG_PATH,
                        filename='tkt_distribution_outside_random_20euro_penalty'
                        )
print("the mean increase of TKT is: {}".format(
    (_tkt_ton_df['TKT_tonkm'].mean()
        - _tkt_ton_df['iter0_TKT_tonkm'].mean())) 
        )


In [ ]:
_tkt_df = all_metrics_random_20euro_af60_df.copy(deep=True)
_tkt_df['TKT_rise'] = _tkt_df['TKT_tonkm'] - _tkt_df['iter0_TKT_tonkm']
_tkt_df['TKT_rise'] = _tkt_df['TKT_rise'] / 1000
_tkt_df = _tkt_df[_tkt_df['TKT_rise'] >= 0]
_tkt_df

In [ ]:
figure_plot.scatter_regression_plot(_tkt_df,
                                    x_col='collaboration_rate',
                                    y_col='TKT_rise',
                                    figure_size=(3.3,2.8),
                                    xlabel='Collaboration rate',
                                    ylabel='TKT rise',
                                    label_size=14,
                                    annotation_fontsize=12,
                                    # title='Scatter plot with regression line: TKT Rise vs Collaboration Rate',
                                    show_corr=True,
                                    add_regression=True,
                                    show_equation=False,
                                    show_r2=True,
                                    scatter_color='#458CBC',
                                    scatter_alpha=0.9,
                                    reg_color='#0b2c60',
                                    # Output
                                    figure_folder=OUTPUT_FIG_PATH,
                                    filename='scatter_regression_tkt_rise_vs_collab_rate_all_random_20euro_af60.png'
                                    )

### Joint plot of collaboration rate, carrier score, receiver score and fleet size.


In [ ]:
_score_df.query("total_receiver_scores >= -825")

In [ ]:
_score_df = all_metrics_random_20euro_af60_df[all_metrics_random_20euro_af60_df['instance'].between(0,100)]
_score_df.query("final_carrier_score >= 0 and total_receiver_scores >= -1000", inplace=True)
_score_df['iter0_total_receiver_scores'] = -1000

_colors = figure_plot.generate_smooth_colors(
    input_colors=["#abd1e9","#0b2c60"],
    n_req_colors=11
)

figure_plot.joint_scatter_plot(
    _score_df,
    x_col='final_carrier_score',
    y_col='total_receiver_scores',
    n_bins=10,
    figure_size=(6, 6),
    marginal_type='both',  # 同时显示 histogram 和 KDE
    marginal_label_size=14,
    xlabel='Carrier score',
    ylabel='Receiver score',
    label_size=16,
    show_corr=False,
    # scatter_color="#8dadc3",
    scatter_alpha=0.9,
    hist_color="#4F81A3",
    kde_linestyle='--',
    kde_alpha=0.9,
    ## the second scatter plot
    # data_df2=_score_df,
    # x_col2='iter0_carrier_score',
    # y_col2='iter0_total_receiver_scores',
    # scatter2_marker='X',
    # scatter2_color='#f6bdb1',
    # scatter2_edgecolor="#f09785",
    # scatter2_alpha=0.9,
    # scatter2_size=90,
    ## Group the main scatter points by collaboration rate
    size_group_col='collaboration_rate',
    size_min=30,
    size_max=300,
    show_size_legend=False,
    # size_legend_loc='lower right',
    ## Group the main scatter points by fleet size
    color_group_col='collaboration_rate',
    # color_group_col='diff_fleet_size',
    cmap_color_list=_colors,
    show_color_legend=False,
    ## Output
    # add_regression=True,
    figure_folder=OUTPUT_FIG_PATH,
    # filename='joint_RC_scores_with_collab_rate_all_random_20euro_af60.png',
)


### Fleet size

In [ ]:
pd.cut(_score_df['collaboration_rate'], bins=[0, 0.4, 0.6, 0.8, 1.0], retbins=True, include_lowest=True)

In [ ]:
_score_df

In [ ]:
# _colors = figure_plot.generate_smooth_colors(
#     input_colors=["#abd1e9", "#0b2c60"],
#     n_req_colors=11
# )
figure_plot.stacked_proportion_plot(_score_df,
                                    figure_size=(3.5,3),
                                     bin_col='collaboration_rate',
                                     value_col='diff_fleet_size',
                                     bin_method='custom',
                                     custom_bins=[0, 0.3, 0.6, 1.0],
                                     xtick_labels=['Low \n(0-0.3)', 'Medium \n(0.4-0.6)', 'High \n(0.7-1.0)'],
                                     xtick_rotation=0,
                                     xtick_ha='center',
                                     xtick_fontsize=12,
                                     colors=["#b6b8b7", "#abd1e9", "#458CBC", "#0b2c60"][::-1],
                                     xlabel='Collaboration rate',
                                     ylabel='Proportion (%)',
                                     label_size=14,
                                     show_counts=True,
                                     count_fontsize=12,
                                     percentage_fontsize=14,
                                     # Legend settings
                                     legend_bbox=(0.5, 1.4),
                                     legend_ncol=3,
                                     legend_title='Fleet size difference',
                                     show_legend=False,
                                     # Output
                                     figure_folder=OUTPUT_FIG_PATH,
                                     filename='stacked_proportion_fleet_size_diff_vs_collab_all_random_af60_pen20.png'
                                     )

In [ ]:
figure_plot.scatter_regression_plot(all_metrics_random_20euro_af60_df,
                                    x_col='collaboration_rate',
                                    y_col='diff_fleet_size',
                                    figure_size=(4,3),
                                    xlabel='Collaboration rate',
                                    ylabel='Fleet size reduction',
                                    label_size=14,
                                    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
                                    show_corr=True,
                                    add_regression=True,
                                    show_equation=False,
                                    show_r2=False,
                                    scatter_color='#458CBC',
                                    scatter_alpha=0.9,
                                    reg_color='#0b2c60',
                                    # Output
                                    figure_folder=OUTPUT_FIG_PATH,
                                    filename='scatter_regression_fleet_size_reduction_vs_collab_rate_all_random_20euro_af60.png'
                                    )

### Total cost savings
Plot the correlations between the collaboration rate and the total cost savings

In [ ]:
figure_plot.scatter_regression_plot(all_metrics_random_20euro_af60_df,
                                    x_col='collaboration_rate',
                                    y_col='total_cost_savings',
                                    figure_size=(3.3,2.8),
                                    xlabel='Collaboration rate',
                                    ylabel='Total cost savings',
                                    label_size=14,
                                    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
                                    show_corr=True,
                                    add_regression=True,
                                    show_equation=False,
                                    show_r2=True,
                                    annotation_fontsize=12,
                                    scatter_color='#458CBC',
                                    scatter_alpha=0.9,
                                    reg_color='#0b2c60',
                                    # Output
                                    figure_folder=OUTPUT_FIG_PATH,
                                    filename='scatter_regression_cost_savings_vs_collab_rate_all_random_20euro_af60.png'
                                    )

## Compare specific low-run and high-run random scenarios

In [ ]:
all_metrics_random_20euro_af60_df.query("collaboration_rate == 0")

In [ ]:
all_metrics_random_20euro_af60_df.query("collaboration_rate == 1")

In [ ]:
_low_collab_ins_id = 15 # or 9
_high_collab_ins_id = 18
_second_high_collab_ins_id = 49

anls_af = 0.6
# _medium_collab_ins_id = 41
# specific_run_list = [_low_collab_ins_id, _high_collab_ins_id, _medium_collab_ins_id]

In [ ]:
low_collab_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH, 
    f'{DepotLocation.INSIDE.value}-{ReceiverDistribution.CLUSTERED.value}-af{anls_af:.02f}-p{0.0056}-i{_low_collab_ins_id:02d}',
    'geo_data.geojson'))
high_collab_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH, 
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.RANDOM.value}-af{anls_af:.02f}-p{0.0056}-i{_high_collab_ins_id:02d}',
    'geo_data.geojson'))    
second_high_collab_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH,
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.RANDOM.value}-af{anls_af:.02f}-p{0.0056}-i{_second_high_collab_ins_id:02d}',
    'geo_data.geojson'))


### Plot spatial distribution

#### High

In [ ]:
''' Plot receiver locations at link midpoints for high collaboration scenario'''
figure_plot.network_locations_plot(
    network=network,
    locations_gdf=high_collab_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=True,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (High Collaboration)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename='receiver_locations_high_collab_outside_random_af60_20euro.png'
)

#### 2nd High

In [ ]:
''' Plot receiver locations at link midpoints for medium collaboration scenario'''

figure_plot.network_locations_plot(
    network=network,
    locations_gdf=second_high_collab_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=True,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (High Collaboration)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename='receiver_locations_second_high_collab_random_af60_20euro.png'
)

#### Low

In [ ]:
''' Plot receiver locations at link midpoints for low collaboration scenario '''
figure_plot.network_locations_plot(
    network=network,
    locations_gdf=low_collab_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=True,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (Low Collaboration)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename='receiver_locations_low_collab_random_af60_20euro.png'
)

### Agg statistics

In [ ]:
com_df.plot.scatter(x='collaboration_rate', y='mean_receiver_dist_to_depot_euclidean_km')


In [ ]:
com_df.plot.scatter(x='collaboration_rate', y='clustering_index_euclidean_km')



In [ ]:
com_df_collab_40_50 = com_df[
    (com_df['collaboration_rate'].between(0.4, 0.6))
]
com_df_collab_60_80 = com_df[
    (com_df['collaboration_rate'].between(0.7, 0.8))
]
com_df_collab_90_100 = com_df[
    (com_df['collaboration_rate'].between(0.7, 1.0))
]

In [ ]:
## Print the mean receiver to depot euclidean distance for each subgroup
print("the mean receiver to depot euclidean distance for subgroup 40-50% is:", com_df_collab_40_50['mean_receiver_dist_to_depot_euclidean_km'].mean())
print("the mean receiver to depot euclidean distance for subgroup 60-80% is:", com_df_collab_60_80['mean_receiver_dist_to_depot_euclidean_km'].mean())
print("the mean receiver to depot euclidean distance for subgroup 90-100% is:", com_df_collab_90_100['mean_receiver_dist_to_depot_euclidean_km'].mean())

print()
## print the mean receiver to depot road distance for each subgroup
print("the mean receiver to depot road distance for subgroup 40-50% is:", com_df_collab_40_50['mean_receiver_dist_to_depot_network_km'].mean())
print("the mean receiver to depot road distance for subgroup 60-80% is:", com_df_collab_60_80['mean_receiver_dist_to_depot_network_km'].mean())
print("the mean receiver to depot road distance for subgroup 90-100% is:", com_df_collab_90_100['mean_receiver_dist_to_depot_network_km'].mean())

print()
## Print the mean clustering_index_euclidean_km for each subgroup
print("the mean clustering_index_euclidean_km for subgroup 40-50% is:", com_df_collab_40_50['clustering_index_euclidean_km'].mean())
print("the mean clustering_index_euclidean_km for subgroup 60-80% is:", com_df_collab_60_80['clustering_index_euclidean_km'].mean())
print("the mean clustering_index_euclidean_km for subgroup 90-100% is:", com_df_collab_90_100['clustering_index_euclidean_km'].mean())

print()
### Print the mean clustering_index_network_km for each subgroup
print("the mean clustering_index_network_km for subgroup 40-50% is:", com_df_collab_40_50['clustering_index_network_km'].mean())
print("the mean clustering_index_network_km for subgroup 60-80% is:", com_df_collab_60_80['clustering_index_network_km'].mean())
print("the mean clustering_index_network_km for subgroup 90-100% is:", com_df_collab_90_100['clustering_index_network_km'].mean())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.boxplot(
    [com_df_collab_40_50['mean_receiver_dist_to_depot_euclidean_km'],
     com_df_collab_60_80['mean_receiver_dist_to_depot_euclidean_km'],
     com_df_collab_90_100['mean_receiver_dist_to_depot_euclidean_km']],
    labels=['40-60%', '70-80%', '90-100%'],
    patch_artist=True,
    boxprops=dict(facecolor='#8dadc3', color='black', alpha=0.7),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black'),
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=5, alpha=0.6)
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.boxplot(
    [com_df_collab_40_50['clustering_index_euclidean_km'],
     com_df_collab_60_80['clustering_index_euclidean_km'],
     com_df_collab_90_100['clustering_index_euclidean_km']],
    labels=['40-60%', '70-80%', '90-100%'],
    patch_artist=True,
    boxprops=dict(facecolor='#8dadc3', color='black', alpha=0.7),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black'),
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=5, alpha=0.6)
)

#### Read and agg NNI metric

In [ ]:
'''read all geo data from clean folder and aggregate NNI metrics'''
nni_df = agg_anls.aggregate_nni_from_clean_folder(
    str(clean_data_path),
    study_area_by_distribution={
        'CLUSTERED': 7000*7000,
        'DISPERSED': 5000*5000,
        'FULLY_RANDOM': 6000 * 6000,
    }
)
nni_df.head(10)

In [ ]:
outside_random_20_af80_nni_df = nni_df[
    (nni_df['depot_location'] == DepotLocation.OUTSIDE.value) &
    (nni_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value) &
    (nni_df['penalty'] == 0.0056) &
    (nni_df['allocation_factor'] == 0.8)
]
# merge with all_metrics_outside_random_20euro_df to get collaboration rate, etc metrics
outside_random_20_af80_nni_df = pd.merge(
    outside_random_20_af80_nni_df,
    all_metrics_outside_random_20euro_af80_df,
    left_on=['instance_id', 'penalty', 'depot_location', 'receiver_distribution', 'allocation_factor'],
    right_on=['instance', 'penalty', 'depot_location', 'receiver_distribution', 'allocation_factor'],
    how='left'
)
outside_random_20_af80_nni_df

In [ ]:
outside_random_20_af80_nni_df.plot.scatter(x='nni',
                                      y='collaboration_rate')

In [ ]:
# Box plot NNI by collaboration_rate subgroups
bins = [0, 0.39, 0.51, 0.8, 1.1]
labels = ['0-0.39', '0.40-0.61', '0.62-0.79', '0.80-1.0']
outside_random_20_af80_nni_df['collab_group'] = pd.cut(
    outside_random_20_af80_nni_df['collaboration_rate'], 
    bins=bins, 
    labels=labels,
    include_lowest=True
)

fig, ax = plt.subplots(figsize=(10, 6))
outside_random_20_af80_nni_df.boxplot(column='nni', by='collab_group', ax=ax)
ax.set_xlabel('Collaboration Rate Group')
ax.set_ylabel('NNI (Nearest Neighbor Index)')
ax.set_title('NNI Distribution by Collaboration Rate Subgroups')
plt.suptitle('')  # Remove automatic title
plt.tight_layout()
plt.show()

# Spatial sensitivity
For this part, we probably need to 
1. first plot the box plots for penalty-sweep and af-sweep across scenarios, to demonstrate/prove the patterns are consistent to the above **cost-benefit sensitivity**.
2. Then, dive into one particular AF-Pen setting group (e.g., af=0.6, pen=0.0056; be consistent with the previous one?). To mainly explore the correlations between spatial index and other key metrics (e.g., total cost savings)
3. Since the clustering index looks irrelevant (little correlated) to other metrics, it is necessary to find another way to demonstrate the power of receivers' spatial distributions.

In [ ]:
ins_spatial_index_df = agg_anls.aggregate_nni_from_clean_folder(
    clean_folder_path=OUTPUT_PATH,
    study_area_by_distribution={
        ReceiverDistribution.CLUSTERED.value: 5000*5000,
        ReceiverDistribution.RANDOM.value: 5000*5000,
        ReceiverDistribution.DISPERSED.value: 9000*9000
    },
    network_node_df=network_nodes,
    network_link_df=network_links,
    network_graph=network_graph,
    verbose=True
)
ins_spatial_index_df

In [ ]:
spatial_cols = ['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'instance_id',
                'nni', 'pattern', 'centroid_to_depot_euclidean_km', 'centroid_to_depot_network_km']

all_metrics_df = pd.merge(
    all_metrics_df,
    ins_spatial_index_df[spatial_cols],
    left_on=['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'instance'],
    right_on=['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'instance_id'],
    how='left'
)
all_metrics_df

In [ ]:
specific_anls_af = 0.6
specific_anls_penalty = 0.0056

In [ ]:
ins_center_clustered_specific_anls_df = all_metrics_df[
    (all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) &
    (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value) &
    (all_metrics_df['allocation_factor'] == specific_anls_af) &
    (all_metrics_df['penalty'] == specific_anls_penalty)
    ]

ins_center_dispersed_specific_anls_df = all_metrics_df[
    (all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) &
    (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value) &
    (all_metrics_df['allocation_factor'] == specific_anls_af) &
    (all_metrics_df['penalty'] == specific_anls_penalty)
    ]

ins_outside_clustered_specific_anls_df = all_metrics_df[
    (all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) &
    (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value) &
    (all_metrics_df['allocation_factor'] == specific_anls_af) &
    (all_metrics_df['penalty'] == specific_anls_penalty)
    ]

ins_outside_dispersed_specific_anls_df = all_metrics_df[
    (all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) &
    (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value) &
    (all_metrics_df['allocation_factor'] == specific_anls_af) &
    (all_metrics_df['penalty'] == specific_anls_penalty)
    ]


### Test for GLM

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:

# df = pd.concat([
#     ins_center_clustered_specific_anls_df,
#     ins_center_dispersed_specific_anls_df,
#     ins_outside_clustered_specific_anls_df,
#     ins_outside_dispersed_specific_anls_df
# ], ignore_index=True)

df = all_metrics_df.query("allocation_factor == 0.6")

# =========
# 2) set up binomial response: k successes out of N trials
# =========
N_TRIALS = 10
df["k"] = df["num_collaborative_receivers"].astype(int)
df["n"] = N_TRIALS
df["y"] = df["k"] / df["n"]  # proportion

# =========
# 3) predictors you mentioned
# depot: 0=centered, 1=outside (in your file: 'center' vs 'left')
# =========
df["depot"] = (df["depot_location"].str.lower() != "center").astype(int)

# choose cost column
# - penalty_std looks like "€/hour" (e.g., 20/35/50)
# - penalty looks like "€/sec" (e.g., 20/3600 = 0.00556)
cost_col = "penalty_std" if "penalty_std" in df.columns else "penalty"
df["cost"] = df[cost_col].astype(float)

# choose distance column (pick one)
dist_col = "centroid_to_depot_network_km"  # or "centroid_to_depot_network_km"
df["dist"] = df[dist_col].astype(float)

df["nni"] = df["nni"].astype(float)

# =========
# 4) safe z-score helper (avoids NaN if a column is constant in your sample)
# =========
def safe_z(series: pd.Series) -> pd.Series:
    mu = float(np.nanmean(series))
    sd = float(np.nanstd(series))
    if sd == 0 or np.isnan(sd):
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - mu) / sd

df["cost_z"] = safe_z(df["cost"])
df["nni_z"]  = safe_z(df["nni"])
df["dist_z"] = safe_z(df["dist"])

# =========
# 5) choose how to treat cost
# If you only have a few discrete cost levels (10/20/35/50), treat it as categorical
# =========
if df["cost"].nunique() <= 6:
    cost_term = "C(cost)"      # categorical (recommended for your boxplot-like behavior)
else:
    cost_term = "cost_z"       # continuous

# =========
# 6) GLM formula (edit here)
# =========
# - cost_term * depot expands to: cost + depot + cost:depot
# - nni_z * depot expands to: nni_z + depot + nni_z:depot
#   (depot is already included, but that's fine; statsmodels handles it)
# formula = f"y ~ {cost_term} * depot + nni_z * depot + dist_z"
formula = f"y ~ {cost_term} * depot * nni_z + dist_z"
# =========
# 7) fit Binomial GLM with freq_weights = number of trials (N=10)
# robust SE (HC1) is helpful if you later worry about mild misspecification
# =========
model = smf.glm(
    formula=formula,
    data=df,
    family=sm.families.Binomial(),
    freq_weights=df["n"],
)

res = model.fit(cov_type="HC1")
print(res.summary())

# =========
# 8) optional: predicted collaboration probability for each instance
# =========
df["p_hat"] = res.predict(df)
print(df[["k", "n", "y", "p_hat", "cost", "depot", "nni", "dist"]].head())

In [ ]:
# ========== 1) choose the cost levels to plot ==========
# Prefer explicit levels (as you requested)
cost_levels = [10, 20, 35, 50]

# If your dataset has different/extra levels, you can uncomment this:
# cost_levels = sorted(df["cost"].unique())

# ========== 2) compute z-score scaling consistent with your fit ==========
# IMPORTANT: match how you created z-scores when fitting.
# In your earlier code you used np.nanstd (ddof=0), so we use ddof=0 here.
cost_mean = float(np.nanmean(df["cost"]))
cost_std  = float(np.nanstd(df["cost"]))  # ddof=0

# If you fitted using existing df["nni_z"] and df["dist_z"], skip these.
# Otherwise compute from raw nni/dist columns.
if "nni_z" in df.columns:
    nni_mean = float(np.nanmean(df["nni_z"]))  # should be ~0
    nni_std  = float(np.nanstd(df["nni_z"]))   # should be ~1
else:
    nni_mean = float(np.nanmean(df["nni"]))
    nni_std  = float(np.nanstd(df["nni"]))

if "dist_z" in df.columns:
    dist_mean = float(np.nanmean(df["dist_z"]))  # should be ~0
    dist_std  = float(np.nanstd(df["dist_z"]))   # should be ~1
else:
    # choose the same dist column you used in fitting
    # e.g., df["centroid_to_depot_euclidean_km"] or df["centroid_to_depot_network_km"]
    dist_col = "dist"
    dist_mean = float(np.nanmean(df[dist_col]))
    dist_std  = float(np.nanstd(df[dist_col]))

def zscore(x, mean, std):
    if std == 0 or np.isnan(std):
        return np.zeros_like(np.asarray(x, dtype=float))
    return (np.asarray(x, dtype=float) - mean) / std

# ========== 3) build prediction grid ==========
# We draw lines for depot in {0(center),1(outside)} and NNI_z in {-1,0,1}
nni_levels_z = [-1.0, 0.0, 1.0]  # low / medium / high (in SD units)

grid = []
for depot in [0, 1]:
    for nni_z_val in nni_levels_z:
        for c in cost_levels:
            row = {
                "cost": c,  # keep original cost for plotting / for C(cost) models
                "cost_z": zscore([c], cost_mean, cost_std)[0],  # for continuous-cost_z models
                "depot": depot,
                "nni_z": nni_z_val,
                "dist_z": 0.0,  # hold distance at mean (z=0). Change if you want.
            }
            grid.append(row)

pred_df = pd.DataFrame(grid)

# If your model used raw nni/dist (not z), provide them too (harmless if unused)
if "nni" in df.columns and "nni_z" not in df.columns:
    pred_df["nni"] = pred_df["nni_z"] * nni_std + nni_mean
if "dist" in df.columns and "dist_z" not in df.columns:
    pred_df["dist"] = pred_df["dist_z"] * dist_std + dist_mean

# ========== 4) prediction + CI ==========
# For models fitted with formula API, passing a DataFrame works (patsy rebuilds X)
pred_res = res.get_prediction(pred_df)
sf = pred_res.summary_frame(alpha=0.05)  # mean, mean_ci_lower, mean_ci_upper, etc.

pred_df = pd.concat([pred_df, sf], axis=1)

# ========== 5) plot ==========
# We'll plot predicted cooperation probability p_hat (= collaboration rate),
# with 95% CI as shaded band.
fig = plt.figure(figsize=(9, 6))

def nice_label(depot, nni_z_val):
    depot_str = "center" if depot == 0 else "outside"
    if nni_z_val == -1:
        nni_str = "low NNI (-1 SD)"
    elif nni_z_val == 0:
        nni_str = "mid NNI (0 SD)"
    else:
        nni_str = "high NNI (+1 SD)"
    return f"{depot_str}, {nni_str}"

for depot in [0, 1]:
    for nni_z_val in nni_levels_z:
        sub = pred_df[(pred_df["depot"] == depot) & (pred_df["nni_z"] == nni_z_val)].copy()
        sub = sub.sort_values("cost")

        x = sub["cost"].to_numpy()
        y = sub["mean"].to_numpy()
        lo = sub["mean_ci_lower"].to_numpy()
        hi = sub["mean_ci_upper"].to_numpy()

        plt.plot(x, y, marker="o", label=nice_label(depot, nni_z_val))
        plt.fill_between(x, lo, hi, alpha=0.15)

plt.xlabel("Cost (original units, e.g., 10/20/35/50)")
plt.ylabel("Predicted collaboration rate (p)")
plt.ylim(-0.02, 1.02)
plt.grid(True, alpha=0.3)
plt.legend(title="Depot & NNI level", frameon=True)
plt.tight_layout()

# Save if you want:
# plt.savefig("predicted_collaboration_curves.png", dpi=300, bbox_inches="tight")
plt.show()

### Test for GAM

In [ ]:
from pygam import LogisticGAM, GammaGAM, s, f, te

In [ ]:
gam_af60_pen20_df = all_metrics_df[(all_metrics_df['penalty'] == 0.0056) &
                                   (all_metrics_df['allocation_factor'] == 0.6) &
                                   (all_metrics_df['receiver_distribution'] != ReceiverDistribution.RANDOM.value)]
gam_af60_df = all_metrics_df[(all_metrics_df['allocation_factor'] == 0.6) &
                             (all_metrics_df['receiver_distribution'] != ReceiverDistribution.RANDOM.value)]

#### Collab. rate

In [ ]:

reg_df = gam_af60_df.copy()

N_TRIALS = 10
reg_df["k"] = reg_df["num_collaborative_receivers"].astype(int)

# depot dummy: 0=center, 1=outside
reg_df["depot"] = (reg_df["depot_location"].str.lower() != "center").astype(int)

# cost
cost_col = "penalty_std" if "penalty_std" in reg_df.columns else "penalty"
reg_df["cost"] = reg_df[cost_col].astype(int)

# distance (pick one)
dist_col = "centroid_to_depot_network_km"  # or "centroid_to_depot_network_km"
reg_df["dist_centroid"] = reg_df[dist_col].astype(float)

reg_df["nni"] = reg_df["nni"].astype(float)

reg_df['mean_rc_dist'] = reg_df['mean_receiver_dist_to_depot_network_km'].astype(float)

# =========
# 1) expand each instance into N_TRIALS binary observations
# =========
X_inst = reg_df[["cost", "depot", "nni", "dist_centroid", "mean_rc_dist"]].to_numpy()
X = np.repeat(X_inst, N_TRIALS, axis=0)

y = np.concatenate([
    np.repeat([1, 0], [k, N_TRIALS - k])
    for k in reg_df["k"].to_numpy()
])

# =========
# 2) standardize continuous columns (cost, nni, dist) to help GAM fitting
# depot stays 0/1
# =========
cont_idx = [0, 2, 3, 4]  # indices of cost, nni, dist_centroid, mean_rc_dist in X
scaler = {}
for j in cont_idx:
    mu = X[:, j].mean()
    sd = X[:, j].std()
    scaler[j] = (mu, sd)
    if sd > 0:
        X[:, j] = (X[:, j] - mu) / sd
    else:
        X[:, j] = 0.0  # if constant in sample



In [ ]:
''' Fit a GAM model with '''
gam = LogisticGAM(
    s(0)          # smooth term for cost
    +f(1)          # factor term for depot (0/1)
    +s(2)           # smooth term for nni
    +s(3)           # smooth term for dist_centroid
    +s(4)           # smooth term for mean_rc_dist
    +te(2, 4)  
    +te(2, 4, by=1)        # interaction between NNI and mean receiver distance to depot, varying by depot
)

fitted_scores = gam.gridsearch(X, y, 
                               progress=False,
                               #return_scores=True
                               )
print(f'fitted_scores: {fitted_scores}')
print(f"gam confidence intervals:\n{gam.confidence_intervals(X)}")
# print(f"gam significance:\n{gam.significance()}")
print("GAM summary:")
print(gam.summary())
print("GAM statistics:")
print(gam.statistics_)


In [ ]:
from matplotlib.colors import TwoSlopeNorm

feature_names = ["cost", 
                 "depot", 
                 "nni", 
                 "dist_centroid", 
                 "mean_rc_dist"
                 ]

def inv_std(x_std, j, scaler):
    mu, sd = scaler[j]
    return x_std * sd + mu

def make_diverging_norm(Z):
    """Create a TwoSlopeNorm centered at 0; fall back to None if Z is constant."""
    vmax = max(abs(Z.min()), abs(Z.max()))
    if vmax == 0:
        return None  # all values are zero, use default norm
    return TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

# 1) 画每个term的 partial dependence（在log-odds尺度）
for term_i in range(len(gam.terms)):
    term = gam.terms[term_i]
    if term.isintercept:
        continue

    XX = gam.generate_X_grid(term=term_i)
    pdep, confi = gam.partial_dependence(term=term_i, X=XX, width=0.95)

    # 找这个term主要对应哪个特征（s(i)/f(i)通常是一个；te是两个或三个）
    idx = term.feature

    if isinstance(idx, (list, tuple, np.ndarray)) and len(idx) == 3:
        # te三维：固定第三个特征的若干切片，每个切片画一张等高线图
        x1 = XX[:, idx[0]]
        x2 = XX[:, idx[1]]
        x3 = XX[:, idx[2]]

        u1 = np.unique(x1)
        u2 = np.unique(x2)
        u3 = np.unique(x3)

        Z_full = pdep.reshape(len(u1), len(u2), len(u3))

        # 反标准化
        x1_plot = inv_std(u1, idx[0], scaler) if idx[0] in scaler else u1
        x2_plot = inv_std(u2, idx[1], scaler) if idx[1] in scaler else u2
        x3_plot = inv_std(u3, idx[2], scaler) if idx[2] in scaler else u3

        # 全局 colorbar 范围
        norm = make_diverging_norm(Z_full)

        # 选取若干切片（最多6张）
        n_slices = min(len(u3), 6)
        slice_indices = np.linspace(0, len(u3) - 1, n_slices, dtype=int)

        fig, axes = plt.subplots(1, n_slices, figsize=(5 * n_slices, 4), squeeze=False)
        for si, s_idx in enumerate(slice_indices):
            ax = axes[0, si]
            Z_slice = Z_full[:, :, s_idx]
            cf = ax.contourf(x2_plot, x1_plot, Z_slice, cmap="RdBu_r", norm=norm, levels=20)
            ax.set_xlabel(feature_names[idx[1]])
            ax.set_ylabel(feature_names[idx[0]])
            ax.set_title(f"{feature_names[idx[2]]}={x3_plot[s_idx]:.2f}")

        fig.suptitle(f"te({idx[0]},{idx[1]},{idx[2]}) partial dependence (log-odds)", y=1.02)
        fig.colorbar(cf, ax=axes.ravel().tolist(), label="effect on log-odds", shrink=0.8)
        plt.tight_layout()
        plt.show()

    elif isinstance(idx, (list, tuple, np.ndarray)) and len(idx) == 2:
        # te二维：画成等高线（partial dependence在log-odds）
        plt.figure()
        x1 = XX[:, idx[0]]
        x2 = XX[:, idx[1]]

        u1 = np.unique(x1)
        u2 = np.unique(x2)
        Z = pdep.reshape(len(u1), len(u2))

        x1_plot = inv_std(u1, idx[0], scaler) if idx[0] in scaler else u1
        x2_plot = inv_std(u2, idx[1], scaler) if idx[1] in scaler else u2

        norm = make_diverging_norm(Z)
        plt.contourf(x2_plot, x1_plot, Z, cmap="RdBu_r", norm=norm, levels=20)
        plt.xlabel(feature_names[idx[1]])
        plt.ylabel(feature_names[idx[0]])
        plt.title(f"te({idx[0]},{idx[1]}) partial dependence (log-odds)")
        plt.colorbar(label="effect on log-odds")
        plt.tight_layout()
        plt.show()

    else:
        # 一维：画曲线 + 置信区间
        plt.figure()
        x = XX[:, idx]
        x_plot = inv_std(x, idx, scaler) if idx in scaler else x

        plt.plot(x_plot, pdep)
        plt.plot(x_plot, confi[:, 0], linestyle="--")
        plt.plot(x_plot, confi[:, 1], linestyle="--")
        plt.xlabel(feature_names[idx])
        plt.ylabel("effect on log-odds")
        plt.title(f"{term} partial dependence (log-odds)")
        plt.tight_layout()
        plt.show()

#### Total cost savings

In [ ]:

reg_df = gam_af60_df.copy()

reg_df["k"] = reg_df["total_cost_savings"].astype(float)

# depot dummy: 0=center, 1=outside
reg_df["depot"] = (reg_df["depot_location"].str.lower() != "center").astype(int)

# cost
cost_col = "penalty_std" if "penalty_std" in reg_df.columns else "penalty"
reg_df["cost"] = reg_df[cost_col].astype(int)

# distance (pick one)
dist_col = "centroid_to_depot_network_km"  # or "centroid_to_depot_network_km"
reg_df["dist_centroid"] = reg_df[dist_col].astype(float)

reg_df["nni"] = reg_df["nni"].astype(float)

reg_df['mean_rc_dist'] = reg_df['mean_receiver_dist_to_depot_network_km'].astype(float)

# =========
# 1) expand each instance into N_TRIALS binary observations
# =========
X = reg_df[["cost", "depot", "nni", "dist_centroid", "mean_rc_dist"]].to_numpy()


y = reg_df["k"].to_numpy()  # total_cost_savings as continuous response for regression GAM

# =========
# 2) standardize continuous columns (cost, nni, dist) to help GAM fitting
# depot stays 0/1
# =========
cont_idx = [0, 2, 3, 4]  # indices of cost, nni, dist_centroid, mean_rc_dist in X
scaler = {}
for j in cont_idx:
    mu = X[:, j].mean()
    sd = X[:, j].std()
    scaler[j] = (mu, sd)
    if sd > 0:
        X[:, j] = (X[:, j] - mu) / sd
    else:
        X[:, j] = 0.0  # if constant in sample



In [ ]:
''' Fit a GammaGAM model — requires y > 0 '''
# GammaGAM needs strictly positive response; filter or shift if needed
mask = y > 0
if not mask.all():
    print(f"WARNING: {(~mask).sum()}/{len(y)} samples have y <= 0 and will be excluded from GammaGAM fit")
X_pos = X[mask]
y_pos = y[mask]

gam = GammaGAM(
    s(0)          # smooth term for cost
    +f(1)          # factor term for depot (0/1)
    +s(2)           # smooth term for nni
    +s(3)           # smooth term for dist_centroid
    +s(4)           # smooth term for mean_rc_dist
    +te(2, 4)        # interaction between NNI and mean receiver distance to depot, varying by depot
    # +te(2, 3)  
    +te(2, 4, by=1)        # interaction between NNI and mean receiver distance to depot, varying by depot
)

# Use gridsearch with higher lambda range to avoid divergence (no need to call fit separately)
fitted_scores = gam.gridsearch(X_pos, y_pos, 
                               lam=np.logspace(-1, 4, 20),
                               progress=False,
                               )
print(f'fitted_scores: {fitted_scores}')
print(f"gam confidence intervals:\n{gam.confidence_intervals(X_pos)}")
print("GAM summary:")
print(gam.summary())
print("GAM statistics:")
print(gam.statistics_)

In [ ]:
from matplotlib.colors import TwoSlopeNorm

feature_names = ["cost", "depot", "nni", "dist_centroid", "mean_rc_dist"]

def inv_std(x_std, j, scaler):
    """反标准化：将标准化后的值还原为原始尺度"""
    mu, sd = scaler[j]
    return x_std * sd + mu

def make_diverging_norm(Z):
    """以0为中心的diverging norm；Z全为常数时返回None"""
    vmax = max(abs(Z.min()), abs(Z.max()))
    if vmax == 0:
        return None
    return TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

# =============================================
# 1) 每个term的partial dependence图（response尺度）
# =============================================
for term_i in range(len(gam.terms)):
    term = gam.terms[term_i]
    if term.isintercept:
        continue

    XX = gam.generate_X_grid(term=term_i)
    pdep, confi = gam.partial_dependence(term=term_i, X=XX, width=0.95)
    idx = term.feature

    if isinstance(idx, (list, tuple, np.ndarray)) and len(idx) == 2:
        # te 二维 → 等高线图
        x1 = XX[:, idx[0]]
        x2 = XX[:, idx[1]]
        u1 = np.unique(x1)
        u2 = np.unique(x2)
        Z = pdep.reshape(len(u1), len(u2))

        x1_plot = inv_std(u1, idx[0], scaler) if idx[0] in scaler else u1
        x2_plot = inv_std(u2, idx[1], scaler) if idx[1] in scaler else u2

        norm = make_diverging_norm(Z)
        plt.figure(figsize=(7, 5))
        plt.contourf(x2_plot, x1_plot, Z, cmap="RdBu_r", norm=norm, levels=20)
        plt.xlabel(feature_names[idx[1]])
        plt.ylabel(feature_names[idx[0]])
        plt.title(f"te({feature_names[idx[0]]}, {feature_names[idx[1]]}) partial dependence")
        plt.colorbar(label="effect on link function")
        plt.tight_layout()
        plt.show()

    else:
        # 一维 → 曲线 + 置信区间
        x = XX[:, idx]
        x_plot = inv_std(x, idx, scaler) if idx in scaler else x

        plt.figure(figsize=(6, 4))
        plt.plot(x_plot, pdep, label="partial dependence")
        plt.fill_between(x_plot, confi[:, 0], confi[:, 1], alpha=0.2, label="95% CI")
        plt.xlabel(feature_names[idx])
        plt.ylabel("effect on link function")
        plt.title(f"{feature_names[idx]} partial dependence (GammaGAM)")
        plt.legend()
        plt.tight_layout()
        plt.show()

# =============================================
# 2) 观测 vs 预测 散点图
# =============================================
y_pred = gam.predict(X_pos)

plt.figure(figsize=(6, 6))
plt.scatter(y_pos, y_pred, alpha=0.5, s=20)
lims = [min(y_pos.min(), y_pred.min()), max(y_pos.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', label="ideal y=x")
plt.xlabel("Observed (total_cost_savings)")
plt.ylabel("Predicted")
plt.title("GammaGAM: Observed vs Predicted")
plt.legend()
plt.tight_layout()
plt.show()

# =============================================
# 3) 残差图
# =============================================
residuals = y_pos - y_pred

plt.figure(figsize=(6, 4))
plt.scatter(y_pred, residuals, alpha=0.5, s=20)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("GammaGAM: Residuals vs Predicted")
plt.tight_layout()
plt.show()

# =============================================
# 4) 残差直方图
# =============================================
plt.figure(figsize=(6, 4))
plt.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
plt.xlabel("Residual")
plt.ylabel("Count")
plt.title("GammaGAM: Residual Distribution")
plt.tight_layout()
plt.show()

#### example code

In [ ]:
# =========
# 3) specify a GAM
# Base additive model:
#   s(cost) + f(depot) + s(nni) + s(dist) + s(mean_rc_dist)
# If you want "cost curve differs by depot", add s(cost, by=depot)
# =========

# --- Model A: additive
gam_A = LogisticGAM(
    s(0, n_splines=20) +       # cost
    f(1) +                    # depot (factor)
    s(2, n_splines=20) +       # nni
    s(3, n_splines=20) +       # dist
    s(4, n_splines=20)         # mean_rc_dist
)

# gridsearch tunes smoothing lambda; good default
fitted_scores = gam_A.gridsearch(X, y, return_scores=True)
print("Model A summary:")
print(gam_A.summary())

# --- Model B: allow cost smooth to vary by depot (interaction-like)
# Interpretation: depot=0 uses baseline s(cost); depot=1 uses s(cost)+s(cost,by=depot)
gam_B = LogisticGAM(
    s(0, n_splines=6) +               # baseline cost smooth
    s(0, by=1, n_splines=6) +         # extra cost smooth for depot=1
    f(1) +                            # depot main effect
    s(2, n_splines=6) +               # nni
    s(3, n_splines=6)                 # dist
)

gam_B.gridsearch(X, y)
print("Model B summary:")
print(gam_B.summary())


gam_C = LogisticGAM(
    s(0, n_splines=6) +                 # cost
    s(0, by=1, n_splines=6) +           # cost extra for depot=1
    f(1) +
    s(2, n_splines=6) +                 # nni
    s(2, by=1, n_splines=6) +           # nni extra for depot=1  (关键)
    s(3, n_splines=6)                   # dist
)
gam_C.gridsearch(X, y)
print("Model C summary:")
print(gam_C.summary())


gam_D = LogisticGAM(
    te(0, 2, n_splines=6) +     # te(cost, nni)
    f(1) +
    s(3, n_splines=6)
)
gam_D.gridsearch(X, y)
print("Model D summary:")
print(gam_D.summary())


# =========
# 4) optional: predict collaboration probability back at instance-level
#    For each instance, predict p for its covariates, then expected collaboration rate = p (since N fixed).
# =========
X0 = df[["cost", "depot", "nni", "dist"]].to_numpy()
# apply same scaling as above
for j in cont_idx:
    mu, sd = scaler[j]
    X0[:, j] = 0.0 if sd == 0 else (X0[:, j] - mu) / sd

df["p_hat_gamB"] = gam_B.predict_mu(X0)  # probability
print(df[["k", "p_hat_gamB", "cost", "depot", "nni", "dist"]].head())

In [ ]:
df.plot.hist(column="total_cost_savings", bins=20, edgecolor="k")

In [ ]:
reg_df.query("total_cost_savings >0").plot.hist(column='total_cost_savings', bins=20, edgecolor='k')

In [ ]:
spatial_concat_df.plot.hist(column='collaboration_rate', bins=10, edgecolor='k')

### Random forest

In [ ]:
N_TRIALS = 10
reg_df = gam_af60_pen20_df.copy()

reg_df["k"] = reg_df["num_collaborative_receivers"].astype(int)
reg_df["depot"] = (reg_df["depot_location"].str.lower() != "center").astype(int)

cost_col = "penalty_std" if "penalty_std" in reg_df.columns else "penalty"
reg_df["cost"] = reg_df[cost_col].astype(float)

reg_df["dist_centroid"] = reg_df["centroid_to_depot_network_km"].astype(float)
reg_df["nni"] = reg_df["nni"].astype(float)

# 你已修复：receiver到depot的平均距离（确保不是跟 dist_centroid 重复）
reg_df["mean_rc_dist"] = reg_df["mean_receiver_dist_to_depot_network_km"].astype(float)

feat_cols = ["cost", "depot", "nni", "dist_centroid", "mean_rc_dist"]
X_inst = reg_df[feat_cols].to_numpy()
k = reg_df["k"].to_numpy().astype(float)

In [ ]:
X = np.repeat(X_inst, 2, axis=0)
y = np.tile([1, 0], len(k)).astype(int)
w = np.vstack([k, N_TRIALS - k]).T.reshape(-1).astype(float)

# 去掉权重为0的行
mask = w > 0
X, y, w = X[mask], y[mask], w[mask]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X, y, sample_weight=w)

In [ ]:
def binomial_loglik(k, n, p):
    p = np.clip(p, 1e-9, 1-1e-9)
    return np.sum(k*np.log(p) + (n-k)*np.log(1-p))

# 对每个instance预测一次就行（用X_inst即可）
p_hat = rf.predict_proba(X_inst)[:, 1]
ll = binomial_loglik(k, N_TRIALS, p_hat)
print("Binomial log-likelihood (higher is better):", ll)

In [ ]:
# 列索引（对应 feat_cols）
COST, DEPOT, NNI, DISTC, MRD = 0, 1, 2, 3, 4

def make_grid_from_data(X_inst, depot_value, n_points=80):
    # 扫 nni 和 mean_rc_dist；其它变量固定为“样本中位数”
    x_ref = np.median(X_inst, axis=0)

    nni_vals = np.linspace(X_inst[:, NNI].min(), X_inst[:, NNI].max(), n_points)
    mrd_vals = np.linspace(X_inst[:, MRD].min(), X_inst[:, MRD].max(), n_points)
    M, N = np.meshgrid(mrd_vals, nni_vals)  # x=mrd, y=nni

    Xg = np.repeat(x_ref.reshape(1,-1), n_points*n_points, axis=0)
    Xg[:, DEPOT] = depot_value
    Xg[:, NNI] = N.reshape(-1)
    Xg[:, MRD] = M.reshape(-1)

    return Xg, nni_vals, mrd_vals

Xg0, nni_vals, mrd_vals = make_grid_from_data(X_inst, depot_value=0, n_points=80)
Xg1, _, _               = make_grid_from_data(X_inst, depot_value=1, n_points=80)

P0 = rf.predict_proba(Xg0)[:,1].reshape(len(nni_vals), len(mrd_vals))
P1 = rf.predict_proba(Xg1)[:,1].reshape(len(nni_vals), len(mrd_vals))
Pd = P1 - P0

vmin = min(P0.min(), P1.min())
vmax = max(P0.max(), P1.max())

plt.figure(figsize=(7,5))
plt.contourf(mrd_vals, nni_vals, P0, vmin=vmin, vmax=vmax)
plt.xlabel("mean_rc_dist")
plt.ylabel("nni")
plt.title("RF: P(collab=1) surface | depot=0")
plt.colorbar(label="probability")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,5))
plt.contourf(mrd_vals, nni_vals, P1, vmin=vmin, vmax=vmax)
plt.xlabel("mean_rc_dist")
plt.ylabel("nni")
plt.title("RF: P(collab=1) surface | depot=1")
plt.colorbar(label="probability")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,5))
plt.contourf(mrd_vals, nni_vals, Pd)
plt.xlabel("mean_rc_dist")
plt.ylabel("nni")
plt.title("RF: Difference surface (depot=1 - depot=0)")
plt.colorbar(label="Δ probability")
plt.tight_layout()
plt.show()

### Box plots of four scenarios' metrics


#### Collab. rate (center-clustered-af0.6)

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_center_clustered_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4.2, 3.6),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af60_all_center_clustered_receivers.png',
)

#### Collab. rate (center-dispersed-af0.6)

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_center_dispersed_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4.2, 3.6),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af60_all_center_dispersed_receivers.png',
)

#### Collab. rate (outside-clustered-af0.6)

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_outside_clustered_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4.2, 3.6),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af60_all_outside_clustered_receivers.png',
)

#### Collab. rate (outside-dispersed-af0.6)

In [ ]:
all_metrics_outside_dispersed_df.query("allocation_factor == 0.6")

In [ ]:
ins_outside_dispersed_specific_anls_df

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_outside_dispersed_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4.2, 3.6),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af60_all_outside_dispersed_receivers.png',
)

### Trial on Moran's I and Getis Ord 

In [ ]:
link_data_path = repo_root / 'data/freightChessboardRC/cleanMoreDispersed'
all_stat_collab_links, all_stat_collab_links_agg = agg_anls.aggregate_collaborative_receivers_by_link(
    str(link_data_path),
    verbose=True
)

In [ ]:
ins_outside_dispersed_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.OUTSIDE.value) &
    (all_stat_collab_links['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)
]

ins_outside_clustered_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.OUTSIDE.value) &
    (all_stat_collab_links['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)
]

ins_center_dispersed_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.INSIDE.value) &
    (all_stat_collab_links['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)
]

ins_center_clustered_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.INSIDE.value) &
    (all_stat_collab_links['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)
]

In [ ]:
ins_center_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.INSIDE.value)
]

ins_outside_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.OUTSIDE.value)
]

In [ ]:
mean_collab_links_center_clustered = spatial_anls.compute_mean_collab_count_by_link(
    ins_center_clustered_specific_stat_collab_links,
    central_area_network_gdf,)
mean_collab_links_center_dispersed = spatial_anls.compute_mean_collab_count_by_link(
    ins_center_dispersed_specific_stat_collab_links,
    full_network_gdf,)
mean_collab_links_outside_clustered = spatial_anls.compute_mean_collab_count_by_link(
    ins_outside_clustered_specific_stat_collab_links,
    central_area_network_gdf,)
mean_collab_links_outside_dispersed = spatial_anls.compute_mean_collab_count_by_link(
    ins_outside_dispersed_specific_stat_collab_links,
    full_network_gdf,)

mean_collab_links_center = spatial_anls.compute_mean_collab_count_by_link(
    ins_center_specific_stat_collab_links,
    central_area_network_gdf,)
mean_collab_links_outside = spatial_anls.compute_mean_collab_count_by_link(
    ins_outside_specific_stat_collab_links,
    full_network_gdf,) 

In [ ]:
mean_collab_links_center.plot(column='total_collab_count',
                                        cmap='Blues',
                                        legend=True,
                                        figsize=(8, 6),
                                        # title='Mean Collaborative Receivers per Link\n(Center - Clustered)',
                                        # edgecolor='k',
                                        linewidth=3)

In [ ]:
mean_collab_links_outside.plot(column='total_collab_count',
                                        cmap='Blues',
                                        legend=True,
                                        figsize=(8, 6),
                                        # title='Mean Collaborative Receivers per Link\n(Center - Clustered)',
                                        # edgecolor='k',
                                        linewidth=3)

In [ ]:
_result = spatial_anls.compute_getis_ord_statistics(
    mean_collab_links_outside, 
    col_name='total_collab_count',
    # weights_type='knn',
    # k_neighbors=5
    weights_type='queen',
    # distance_threshold=1500,
    # binary=True
)

In [ ]:
spatial_anls.compute_getis_ord_statistics(
    mean_collab_links_center, 
    col_name='total_collab_count',
    # weights_type='knn',
    # k_neighbors=5
    weights_type='queen',
    # distance_threshold=1500,
    # binary=True
)

In [ ]:
result_moran_ = spatial_anls.compute_moran_statistics(
    mean_collab_links_center,
     weights_type='queen', 
    col_name='total_collab_count'
)

In [ ]:
result_moran_ = spatial_anls.compute_moran_statistics(
    mean_collab_links_outside,
     weights_type='queen', 
    col_name='total_collab_count'
)

### Map and locations

In [ ]:
_sel_ins = 6
dispersed_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH, 
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.DISPERSED.value}-af{ALLOCATION_FACTOR:.02f}-p{0.0056}-i{_sel_ins:02d}',
    'geo_data.geojson'))
clustered_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH,
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.CLUSTERED.value}-af{ALLOCATION_FACTOR:.02f}-p{0.0056}-i{_sel_ins:02d}',
    'geo_data.geojson'))

In [ ]:
''' Plot the dispersed scenario receiver locations at link midpoints '''
figure_plot.network_locations_plot(
    network=network,
    locations_gdf=dispersed_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=False,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (Dispersed)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename=f'receiver_locations_dispersed_outside_random_af{specific_anls_af:.02f}_p{specific_anls_penalty:.04f}_i{_sel_ins:02d}.png'
)

In [ ]:
''' Plot the clustered scenario receiver locations at link midpoints '''
figure_plot.network_locations_plot(
    network=network,
    locations_gdf=clustered_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=True,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (Clustered)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename=f'receiver_locations_clustered_outside_random_af{specific_anls_af:.02f}_p{specific_anls_penalty:.04f}_i{_sel_ins:02d}.png'
)

### Collaboration rate

In [ ]:
''' Box plot using figure_plot module '''
del_ins = (-1, -50)
# Example usage with the new box_plot function
fig, ax, bp = figure_plot.box_plot(
    data_list=[
        ins_center_clustered_specific_anls_df[~ins_center_clustered_specific_anls_df['instance'].between(del_ins[0], del_ins[1])],
        ins_center_dispersed_specific_anls_df[~ins_center_dispersed_specific_anls_df['instance'].between(del_ins[0], del_ins[1])],
        ins_outside_clustered_specific_anls_df[~ins_outside_clustered_specific_anls_df['instance'].between(del_ins[0], del_ins[1])],
        ins_outside_dispersed_specific_anls_df[~ins_outside_dispersed_specific_anls_df['instance'].between(del_ins[0], del_ins[1])]
    ],
    col_name='collaboration_rate',
    labels=['Center-Clustered', 'Center-Dispersed', 'Outside-Clustered', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#89ADA8", "#ddd2e9c4", "#345e55", "#8e76a6"],
    # Background colors for each box region
    bg_colors=["#CBE6D9", "#f7e0fb", "#9ac99a", "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    # filename=f'boxplot_collaboration_rate_across_spatial_distributions_scenarios_af{specific_anls_af:.02f}_p{specific_anls_penalty:.04f}_i{_sel_ins:02d}.png'
)

It seems not necessary to compare VKT, TT and TKT, 
since the scenarios with the outside depot have way higher values than others.

### VKT

In [ ]:
''' Box plot '''
fig, ax, bp = figure_plot.box_plot(
    data_list=[
        ins_center_clustered_specific_anls_df,
        ins_outside_clustered_specific_anls_df,
        ins_center_dispersed_specific_anls_df,
        ins_outside_dispersed_specific_anls_df
    ],
    col_name='VKT_km',
    labels=['Center-Clustered', 'Outside-Clustered', 'Center-Dispersed', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#89ADA8",  "#345e55", "#ddd2e9c4", "#8e76a6"],
    # Background colors for each box region
    bg_colors=["#CBE6D9", "#9ac99a", "#f7e0fb",  "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='VKT',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    #filename=f'boxplot_vkt_across_spatial_distributions_scenarios_af_{specific_anls_af}.png'
)

In [ ]:
''' Box plot '''
fig, ax, bp = figure_plot.box_plot(
    data_list=[
        ins_center_clustered_specific_anls_df,
        ins_outside_clustered_specific_anls_df,
        ins_center_dispersed_specific_anls_df,
        ins_outside_dispersed_specific_anls_df
    ],
    col_name='VKT_km',
    labels=['Center-Clustered', 'Outside-Clustered', 'Center-Dispersed', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#83A7BE"] * 4,
    # Background colors for each box region
    #bg_colors=["#CBE6D9", "#9ac99a", "#f7e0fb",  "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='VKT',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    #filename=f'boxplot_vkt_across_spatial_distributions_scenarios_af_{specific_anls_af}.png'
)

### VTT

In [ ]:
''' Box plot '''
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(
    [ins_center_clustered_specific_anls_df['VTT_seconds'],
     ins_center_dispersed_specific_anls_df['VTT_seconds'],
     ins_outside_clustered_specific_anls_df['VTT_seconds'],
     ins_outside_dispersed_specific_anls_df['VTT_seconds']],
    labels=['Center-Clustered', 'Center-Dispersed', 'Outside-Clustered', 'Outside-Dispersed'],
    patch_artist=True,
    boxprops=dict(facecolor='#8dadc3', color='black', alpha=0.7),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black'),
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=5, alpha=0.6)
)
ax.set_title(f'VTT Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})')
ax.set_ylabel('VTT (seconds)')
plt.tight_layout()

plt.show()

### TKT

In [ ]:
''' Box plot '''
_ins_center_clustered_specific_anls_df = ins_center_clustered_specific_anls_df.copy()
_ins_center_clustered_specific_anls_df['TKT_tonkm'] = _ins_center_clustered_specific_anls_df['TKT_tonkm'] / 1000
_ins_center_dispersed_specific_anls_df = ins_center_dispersed_specific_anls_df.copy()
_ins_center_dispersed_specific_anls_df['TKT_tonkm'] = _ins_center_dispersed_specific_anls_df['TKT_tonkm'] / 1000
_ins_outside_clustered_specific_anls_df = ins_outside_clustered_specific_anls_df.copy()
_ins_outside_clustered_specific_anls_df['TKT_tonkm'] = _ins_outside_clustered_specific_anls_df['TKT_tonkm'] / 1000
_ins_outside_dispersed_specific_anls_df = ins_outside_dispersed_specific_anls_df.copy()
_ins_outside_dispersed_specific_anls_df['TKT_tonkm'] = _ins_outside_dispersed_specific_anls_df['TKT_tonkm'] / 1000

fig, ax, bp = figure_plot.box_plot(
    data_list=[
        _ins_center_clustered_specific_anls_df,
        _ins_center_dispersed_specific_anls_df,
        _ins_outside_clustered_specific_anls_df,
        _ins_outside_dispersed_specific_anls_df
    ],
    col_name='TKT_tonkm',
    labels=['Center-Clustered', 'Center-Dispersed', 'Outside-Clustered', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#89ADA8", "#ddd2e9c4", "#345e55", "#8e76a6"],
    # Background colors for each box region
    bg_colors=["#CBE6D9", "#f7e0fb", "#9ac99a", "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Ton-km travelled',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    filename=f'boxplot_tkt_across_spatial_distributions_scenarios_af_{specific_anls_af}.png'
)

In [ ]:
''' Box plot '''
_ins_center_clustered_specific_anls_df = ins_center_clustered_specific_anls_df.copy()
_ins_center_clustered_specific_anls_df['TKT_tonkm'] = _ins_center_clustered_specific_anls_df['TKT_tonkm'] / 1000
_ins_center_dispersed_specific_anls_df = ins_center_dispersed_specific_anls_df.copy()
_ins_center_dispersed_specific_anls_df['TKT_tonkm'] = _ins_center_dispersed_specific_anls_df['TKT_tonkm'] / 1000
_ins_outside_clustered_specific_anls_df = ins_outside_clustered_specific_anls_df.copy()
_ins_outside_clustered_specific_anls_df['TKT_tonkm'] = _ins_outside_clustered_specific_anls_df['TKT_tonkm'] / 1000
_ins_outside_dispersed_specific_anls_df = ins_outside_dispersed_specific_anls_df.copy()
_ins_outside_dispersed_specific_anls_df['TKT_tonkm'] = _ins_outside_dispersed_specific_anls_df['TKT_tonkm'] / 1000

fig, ax, bp = figure_plot.box_plot(
    data_list=[
        _ins_center_clustered_specific_anls_df,
        _ins_outside_clustered_specific_anls_df,
        _ins_center_dispersed_specific_anls_df,
        _ins_outside_dispersed_specific_anls_df
    ],
    col_name='TKT_tonkm',
    labels=['Center-Clustered',  'Outside-Clustered', 'Center-Dispersed', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#83A7BE"] * 4,
    # Background colors for each box region
    #bg_colors=["#CBE6D9", "#f7e0fb", "#9ac99a", "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Ton-km travelled',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    #filename=f'boxplot_tkt_across_spatial_distributions_scenarios_af_{specific_anls_af}.png'
)

### Fleet size

In [ ]:
fleet_size_data_dict = {
    'Center-Clustered': ins_center_clustered_specific_anls_df,
    'Center-Dispersed': ins_center_dispersed_specific_anls_df,
    'Outside-Clustered': ins_outside_clustered_specific_anls_df,
    'Outside-Dispersed': ins_outside_dispersed_specific_anls_df
}

In [ ]:
fig, ax = figure_plot.quadrant_donut_chart(
    data_dict=fleet_size_data_dict,
    value_col='final_fleet_size',
    hole_radius=0.4,
    xlabel_right='Dispersed →',
    xlabel_left='← Clustered',
    ylabel_top='↑ Center',
    ylabel_bottom='↓ Outside',
    explode=0.03,
    dpi=350,
    figsize=(6, 6),
)

In [ ]:


fig, ax = figure_plot.nested_donut_chart(
    data_dict,
    group_col='final_fleet_size',
    # title='Fleet Size Distribution by Scenario',
    inner_colors=["#88ABA7", "#ddd2e9c4", "#4d8075", "#b7a4ca"],
    outer_color_list=["#dcbdc3", "#c18d97", "#844954CF", "#844954"],
    outer_cmap='coolwarm',
    show_inner_labels=False,
    show_inner_pct=False,
    show_outer_count=False,
    show_outer_labels=False,
    hole_radius=0.4,
    inner_label_size=5,
    dpi=350,
    outer_label_size=10,
    figsize=(5, 5),
    show_legend=False,
    figure_folder=OUTPUT_FIG_PATH,
    filename='nested_donut_fleet_size_by_scenario.png',
     transparent_bg=True
)

### Spatial index
Relations: (mean-dist-to-depot; clustering index; ) <-> (VKT, TT, TKT, Cost savings, scores, etc)

In [ ]:
spatial_concat_df = pd.concat([
    ins_center_clustered_specific_anls_df,
    ins_center_dispersed_specific_anls_df,
    ins_outside_clustered_specific_anls_df,
    ins_outside_dispersed_specific_anls_df
], ignore_index=True)

In [ ]:
spatial_concat_df['class'] = spatial_concat_df.apply(lambda row: f"{row['depot_location']}_{row['receiver_distribution']}", axis=1)
spatial_concat_df

In [ ]:
spatial_concat_df.groupby('class')['clustering_index_network_km'].mean()

In [ ]:
spatial_concat_df.groupby('class')['clustering_index_euclidean_km'].mean()

In [ ]:
spatial_concat_df.groupby('class')['mean_receiver_dist_to_depot_network_km'].mean()

In [ ]:
spatial_concat_df.groupby('class')['mean_receiver_dist_to_depot_euclidean_km'].mean()

In [ ]:

figure_plot.joint_scatter_plot(
    data_df=spatial_concat_df,
    x_col='mean_receiver_dist_to_depot_euclidean_km',
    y_col='nni',
    # size_group_col='total_cost_savings',
    size_group_col='collaboration_rate',
    color_group_col='class',
    show_size_legend=False,

)

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='clustering_index_euclidean_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df,
    figure_size=(5.8,4),
    xlabel='Disperse index (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=False,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    # figure_folder=OUTPUT_FIG_PATH,
    # filename='scatter_regression_cost_savings_vs_clustering_index.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='mean_receiver_dist_to_depot_euclidean_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df,
    figure_size=(5.8,4),
    xlabel='Mean receiver distance to depot (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_mean_receiver_dist2Depot.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='mean_receiver_dist_to_depot_network_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df,
    figure_size=(5.8,4),
    xlabel='Mean receiver distance to depot (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_mean_receiver_dist2Depot.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='nni',
    y_col='total_cost_savings',
    data_df=spatial_concat_df[spatial_concat_df['pattern'] != 'random'],
    figure_size=(5.8,4),
    xlabel='Nearest Neighbor Index (NNI)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_nni.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_euclidean_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df,
    figure_size=(5.8,4),
    xlabel='Centroid to depot distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_euclidean_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df[spatial_concat_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value],
    figure_size=(5.8,4),
    xlabel='Centroid to depot distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_euclidean_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df[spatial_concat_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value],
    figure_size=(5.8,4),
    xlabel='Centroid to depot distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_network_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df,
    figure_size=(5,4),
    xlabel='Centroid to depot network distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    y_tick_step=100,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot_network.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_network_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df.query("total_cost_savings > 0"),
    figure_size=(5,4),
    xlabel='Centroid to depot network distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    y_tick_step=100,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot_network.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_network_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df[spatial_concat_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value],
    figure_size=(5.8,4),
    xlabel='Centroid to depot distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_euclidean_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df[spatial_concat_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value],
    figure_size=(5.8,4),
    xlabel='Centroid to depot distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot.png'
    ) 